## import module

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import ConcatDataset
from torch.utils.data import random_split
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Model design

In [ ]:
class PCAProjector(nn.Module):
    def __init__(self, input_dim, output_dim=3):
        super(PCAProjector, self).__init__()
        self.linear = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.linear(x)  # (batch, seq_len, output_dim)

class MultiHeadConv(nn.Module):
    def __init__(self, input_dim, output_dim, num_heads=4):
        super(MultiHeadConv, self).__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=input_dim, out_channels=output_dim, kernel_size=ks, padding=ks//2)
            for ks in [3, 5, 7][:num_heads]
        ])
        self.output_dim = output_dim * len(self.convs)

    def forward(self, x):
        x = x.permute(0, 2, 1)  # (batch, input_dim, seq_len)
        out = [conv(x) for conv in self.convs] 
        out = torch.cat(out, dim=1)  # (batch, out_dim * num_heads, seq_len)
        out = out.permute(0, 2, 1)  # (batch, seq_len, out_dim * num_heads)
        return out

class AttentionReduce(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(AttentionReduce, self).__init__()
        self.fc = nn.Linear(input_dim, output_dim)
        self.attn = nn.Linear(input_dim, 1)

    def forward(self, x):
        attn_weights = torch.softmax(self.attn(x), dim=1)  # (batch, seq_len, 1)
        x_weighted = (x * attn_weights).sum(dim=1)  # (batch, input_dim)
        return self.fc(x_weighted)  # (batch, output_dim)

class DisEst(nn.Module):
    def __init__(self, classes=5, feat_dim=52, pca_out=3, lstm_hidden=64, conv_out=32, num_heads=3, final_dim=128):
        super(DisEst, self).__init__()
        self.pca = PCAProjector(feat_dim, pca_out)
        self.lstm = nn.LSTM(pca_out, lstm_hidden, batch_first=True)
        self.time_reduce = nn.Linear(lstm_hidden, final_dim)
        self.drop = nn.Dropout(0.6)

        self.multi_conv = MultiHeadConv(feat_dim, conv_out, num_heads)
        self.attn_reduce = AttentionReduce(conv_out * num_heads, final_dim)
        
        self.identity_branch = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU()
        )

        self.dispelling_branch = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU()
        )
        self.classifier = nn.Linear(1024, 1)

        self.decoder = nn.Sequential(
            nn.Linear(1024 * 2, 2048),
            nn.ReLU(),
            nn.Linear(2048, 200 * 104),
            nn.Tanh()
        )

    def extract_time(self, x):
        x = self.pca(x)
        x, _ = self.lstm(x)
        x = self.drop(x)
        return self.time_reduce(x[:, -1, :]) 

    def extract_space(self, x):
        x = self.multi_conv(x)
        x = self.drop(x)
        return self.attn_reduce(x)
    
    def forward(self, x):
        amp = x[:, :, :52]  #  (batch, 200, 52)
        pha = x[:, :, 52:]  #  (batch, 200, 52)

        f1 = self.extract_time(amp)
        f2 = self.extract_space(amp)
        f3 = self.extract_time(pha)
        f4 = self.extract_space(pha)

        f = torch.cat([f1, f2, f3, f4], dim=1)  # (batch, final_dim * 4)
        f = self.drop(f)
        features = f.to(device)
        identity_features = self.identity_branch(features)  #self.identity_branch(features)
        dispelled_features = self.dispelling_branch(features)
        combined_features = torch.cat([identity_features, dispelled_features], dim=1)
        reconstructed = self.decoder(combined_features).view(-1, 200, 104)
        identity_logits = self.classifier(identity_features) 
        dispelled_logits = self.classifier(dispelled_features)  
        return identity_logits, dispelled_logits, reconstructed
        

## Load data

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, data, labels):
        """
        Args:
            data: numpy (num_samples, num_features)
            labels: numpy (num_samples,)
        """
        self.data = torch.FloatTensor(data) 
        self.labels = torch.LongTensor(labels) if labels.dtype == int else torch.FloatTensor(labels)
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]
    
loaded0 = np.load('location/dataset_drone_10.npz')
loaded1 = np.load('location/dataset_drone_11.npz')
loaded2 = np.load('location/dataset_drone_12.npz')
loaded3 = np.load('location/dataset_drone_13.npz')
loaded4 = np.load('location/dataset_drone_14.npz')
loaded5 = np.load('location/dataset_drone_15.npz')

loaded10 = np.load('location/dataset_drone2_15.npz')
loaded11 = np.load('location/dataset_drone3_15.npz')
loaded12 = np.load('location/dataset_drone4_15.npz')
loaded13 = np.load('location/dataset_drone5_15.npz')

data0, labels0 = loaded0['data'], loaded0['labels']
data1, labels1 = loaded1['data'], loaded1['labels']
data2, labels2 = loaded2['data'], loaded2['labels']
data3, labels3 = loaded3['data'], loaded3['labels']
data4, labels4 = loaded4['data'], loaded4['labels']
data5, labels5 = loaded5['data'], loaded5['labels']
data10, labels10 = loaded10['data'], loaded10['labels']
data11, labels11 = loaded11['data'], loaded11['labels']
data12, labels12 = loaded12['data'], loaded12['labels']
data13, labels13 = loaded13['data'], loaded13['labels']

batch_size = 32

dataset6= CustomDataset(data10,labels10)
dataset7= CustomDataset(data11,labels11)
dataset8= CustomDataset(data12,labels12)
dataset9= CustomDataset(data13,labels13)
test_loader6 = DataLoader(dataset6, batch_size=batch_size, shuffle=False)
test_loader7 = DataLoader(dataset7, batch_size=batch_size, shuffle=False)
test_loader8 = DataLoader(dataset8, batch_size=batch_size, shuffle=False)
test_loader9 = DataLoader(dataset9, batch_size=batch_size, shuffle=False)

data_train = np.concatenate([data0[:420,:,:],data1[:420,:,:], data2[:420,:,:], data3[:420,:,:], data4[:420,:,:], data5[:300,:,:]], axis=0)
labels_train = np.concatenate([labels0[:420],labels1[:420], labels2[:420], labels3[:420], labels4[:420], labels5[:300]], axis=0)
train_set = CustomDataset(data_train,labels_train)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0)    
data_test= np.concatenate([data0[420:,:,:],data1[420:,:,:], data2[420:,:,:], data3[420:,:,:], data4[420:,:,:], data5[300:,:,:]], axis=0)
labels_test = np.concatenate([labels0[420:], labels1[420:], labels2[420:], labels3[420:], labels4[420:], labels5[300:]], axis=0)
test_set = CustomDataset(data_test,labels_test)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

data_test_0= data0[420:,:,:]
labels_test_0 = labels0[420:]
test_set_0 = CustomDataset(data_test_0,labels_test_0)
test_loader_0 = DataLoader(test_set_0, batch_size=batch_size, shuffle=False)
data_test_1= data1[420:,:,:]
labels_test_1 = labels1[420:]
test_set_1 = CustomDataset(data_test_1,labels_test_1)
test_loader_1 = DataLoader(test_set_1, batch_size=batch_size, shuffle=False)
data_test_2= data2[420:,:,:]
labels_test_2 = labels2[420:]
test_set_2 = CustomDataset(data_test_2,labels_test_2)
test_loader_2 = DataLoader(test_set_2, batch_size=batch_size, shuffle=False)
data_test_3= data3[420:,:,:]
labels_test_3 = labels3[420:]
test_set_3 = CustomDataset(data_test_3,labels_test_3)
test_loader_3 = DataLoader(test_set_3, batch_size=batch_size, shuffle=False)
data_test_4= data4[420:,:,:]
labels_test_4 = labels4[420:]
test_set_4 = CustomDataset(data_test_4,labels_test_4)
test_loader_4 = DataLoader(test_set_4, batch_size=batch_size, shuffle=False)
data_test_5= data5[300:,:,:]
labels_test_5 = labels5[300:]
test_set_5 = CustomDataset(data_test_5,labels_test_5)
test_loader_5 = DataLoader(test_set_5, batch_size=batch_size, shuffle=False)

print("data has loaded")

## Train and test

In [ ]:
def train_dis(model, train_loader, p1,p2,p3,num_epochs,lr=0.001):
    criterion_recon = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for data, labels in progress_bar:
            data  = data.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            
            identity_logits, dispelled_logits, reconstructed = model(data)
            loss_id = criterion_recon(identity_logits, labels)
            loss_disid = criterion_recon(dispelled_logits.detach(), labels)
            loss_recon = criterion_recon(reconstructed, data)
            uniform_dist = torch.ones_like(dispelled_logits) / dispelled_logits.size(1)
            loss_H = F.kl_div(F.log_softmax(dispelled_logits, dim=1),uniform_dist + 1e-10,reduction='batchmean')
            loss = loss_id + p1*loss_recon+p2*loss_H -p3*loss_disid 
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            running_loss += loss.item()
            progress_bar.set_postfix(loss=running_loss / len(train_loader))
    
    print("Training Complete")
    
def test_dis(model, test_loader):
    model.eval()
    total_mae = 0.0
    total_samples = 0
    error_list = [] 
    
    with torch.no_grad():
        for data, labels in test_loader:
            data, labels = data.to(device), labels.to(device)
            identity_logits, _, _ = model(data)
            
            batch_size = data.size(0)
            total_samples += batch_size
            total_mae += torch.sum(torch.abs(identity_logits - labels)).item()
            batch_errors = torch.abs(identity_logits - labels).cpu().numpy()
            error_list.extend(batch_errors.flatten().tolist())
    
    overall_mae = total_mae / total_samples
    return overall_mae, np.array(error_list)

## Evaluation

In [ ]:
model = DisEst().to(device)
p1=0.2296
p2=0.6057
p3=0.1207
train_dis(model, train_loader,p1,p2,p3,num_epochs=15)
val1, error_list1 = test_dis(model, test_loader)
val2, error_list2 = test_dis(model, test_loader_0)
val3, error_list3 = test_dis(model, test_loader_1)
val4, error_list4 = test_dis(model, test_loader_2)
val5, error_list5 = test_dis(model, test_loader_3)
val6, error_list6 = test_dis(model, test_loader_4)
val7, error_list7 = test_dis(model, test_loader_5)
val8, error_list8 = test_dis(model, test_loader6)
val9, error_list9 = test_dis(model, test_loader7)
val10, error_list10 = test_dis(model, test_loader8)
val11, error_list11 = test_dis(model, test_loader9)
print(f"error: {val1:.4}m，{val2:.4}m，{val3:.4}m，{val4:.4}m，{val5:.4}m，{val6:.4}m，{val7:.4}m")
print(f"error: {val8:.4}m，{val9:.4}m，{val10:.4}m，{val11:.4}m")